In [4]:
!pip install ultralytics opencv-python matplotlib PyYAML

In [5]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")

CUDA available: True


In [6]:
import os

dataset_root = os.getcwd() 

for root, dirs, files in os.walk(dataset_root):
    for f in files:
        ext = os.path.splitext(f)[1]
        if ext in [".json", ".xml", ".txt"]:
            print(os.path.join(root, f))
            break 

c:\Users\A\Documents\ML2\Project\test\_annotations.coco.json
c:\Users\A\Documents\ML2\Project\test\labels\2318424-17_jpg.rf.4dd7c5bd6a22f4d28fe23d71a9516211.txt
c:\Users\A\Documents\ML2\Project\train\_annotations.coco.json
c:\Users\A\Documents\ML2\Project\train\labels\2268363-22_jpg.rf.c648482be70f6121f748ad7ee5b63204.txt
c:\Users\A\Documents\ML2\Project\valid\_annotations.coco.json
c:\Users\A\Documents\ML2\Project\valid\labels\2302584-15_png.rf.a709a579bf37fd27000fd7c5da1162c6.txt


# Data

Dataset: https://www.kaggle.com/datasets/naufalahnaf17/manga-text-detection

In [7]:
import json
from pathlib import Path

# Maps COCO category id → YOLO class index
CATEGORY_MAP = {0: 0, 1: 1}

def convert_coco_to_yolo(json_path, output_label_dir):
    """
    Reads a COCO _annotations_coco.json and writes one .txt per image
    into output_label_dir in YOLO format.
    """
    output_label_dir = Path(output_label_dir)
    output_label_dir.mkdir(parents=True, exist_ok=True)

    with open(json_path, "r") as f:
        data = json.load(f)

    # Build a lookup: image_id → {file_name, width, height}
    image_info = {
        img["id"]: img for img in data["images"]
    }

    # Group annotations by image_id
    annotations_by_image = {}
    for ann in data["annotations"]:
        img_id = ann["image_id"]
        annotations_by_image.setdefault(img_id, []).append(ann)

    converted = 0
    for img_id, img_meta in image_info.items():
        img_w = img_meta["width"]
        img_h = img_meta["height"]
        stem  = Path(img_meta["file_name"]).stem
        anns  = annotations_by_image.get(img_id, [])

        lines = []
        for ann in anns:
            cat_id = ann["category_id"]
            if cat_id not in CATEGORY_MAP:
                continue

            yolo_class = CATEGORY_MAP[cat_id]
            x_min, y_min, w, h = ann["bbox"]  # COCO format

            # Convert to YOLO normalized cx, cy, w, h
            x_center = (x_min + w / 2) / img_w
            y_center  = (y_min + h / 2) / img_h
            norm_w    = w / img_w
            norm_h    = h / img_h

            lines.append(f"{yolo_class} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}")

        # Write label file (even if empty — means no objects in image)
        out_file = output_label_dir / (stem + ".txt")
        out_file.write_text("\n".join(lines))
        converted += 1

    print(f"✓ {json_path} → {converted} label files written to {output_label_dir}")


# Run for all three splits
for split in ["train", "valid", "test"]:
    convert_coco_to_yolo(
        json_path         = f"{split}/_annotations.coco.json",
        output_label_dir  = f"{split}/labels"
    )

✓ train/_annotations.coco.json → 315 label files written to train\labels
✓ valid/_annotations.coco.json → 90 label files written to valid\labels
✓ test/_annotations.coco.json → 45 label files written to test\labels


In [8]:
import shutil
from pathlib import Path

for split in ["train", "valid", "test"]:
    split_path = Path(split)
    images_dir = split_path / "images"
    images_dir.mkdir(exist_ok=True)

    # Move all image files into the images/ subfolder
    for img in split_path.glob("*.jpg"):
        shutil.move(str(img), images_dir / img.name)
    for img in split_path.glob("*.png"):
        shutil.move(str(img), images_dir / img.name)

    print(f"✓ {split}: {len(list(images_dir.glob('*')))} images moved")

✓ train: 315 images moved
✓ valid: 90 images moved
✓ test: 45 images moved


In [9]:
import yaml
from pathlib import Path

config = {
    "path"  : str(Path(os.getcwd()).resolve()),
    "train" : "train/images",
    "val"   : "valid/images",
    "test"  : "test/images",
    "nc"    : 2,
    "names" : ["words", "textrotation"]
}

with open("data.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(Path("data.yaml").read_text())

names:
- words
- textrotation
nc: 2
path: C:\Users\A\Documents\ML2\Project
test: test/images
train: train/images
val: valid/images



In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

results = model.train(
    data     = "data.yaml",
    epochs   = 100,
    imgsz    = 512,       
    batch    = 16,        
    device   = 0,
    patience = 20,        # early stopping if no improvement for 20 epochs
    lr0      = 0.001,
    mosaic   = 1.0,
    fliplr   = 0.5,
    flipud   = 0.0,
    degrees  = 0,
    project  = "manga_runs",
    name     = "bubble_detector",
)

Ultralytics 8.4.37  Python-3.11.0 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=bubble_detector, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pat

Loading Model

In [ ]:
from ultralytics import YOLO

model = YOLO("runs/detect/manga_runs/bubble_detector/weights/best.pt")

results = model.predict(
    source = "test/images/31841-3_jpg.rf.2487413c8d2e95f9ff58c5412339d5be.jpg",
    conf   = 0.4,
    save   = True
)

for result in results:
    for box in result.boxes:
        print(f"Class: {result.names[int(box.cls)]}, Confidence: {box.conf.item():.2f}")
        print(f"Bbox: {box.xyxy[0].tolist()}") 


image 1/1 c:\Users\A\Documents\ML2\Project\test\images\31841-3_jpg.rf.2487413c8d2e95f9ff58c5412339d5be.jpg: 512x512 7 textrotations, 74.3ms
Speed: 2.0ms preprocess, 74.3ms inference, 12.5ms postprocess per image at shape (1, 3, 512, 512)
Results saved to C:\Users\A\Documents\ML2\Project\runs\detect\predict
Class: textrotation, Confidence: 0.89
Bbox: [25.351425170898438, 342.4648132324219, 105.42106628417969, 412.3081970214844]
Class: textrotation, Confidence: 0.86
Bbox: [376.2115478515625, 19.141159057617188, 447.99249267578125, 90.84336853027344]
Class: textrotation, Confidence: 0.85
Bbox: [35.42279052734375, 191.7061767578125, 103.20120239257812, 259.70263671875]
Class: textrotation, Confidence: 0.82
Bbox: [230.391845703125, 190.11480712890625, 274.209228515625, 254.73422241210938]
Class: textrotation, Confidence: 0.74
Bbox: [412.1322326660156, 462.02093505859375, 471.4217834472656, 500.75555419921875]
Class: textrotation, Confidence: 0.62
Bbox: [426.840576171875, 357.457275390625, 